# Project Main: Store Clean Tables in DuckDB

In Day 1 Step 02, we cleaned the API data into three area-level CSV files:

- Weather forecast by 30-minute timestamp and area
- Taxi count by 30-minute timestamp and area
- Rainfall mm by 30-minute timestamp and area

Now we put those tables into DuckDB and learn SQL.

The main ideas are:

1. A relational database stores data in tables.
2. Tables contain rows, columns, and values.
3. Keys help us identify and connect rows.
4. SQL lets us filter, aggregate, and join data.
5. Keeping staging tables improves auditability.
6. Transformations can happen before loading data, or inside the database.

## 1. Database Concepts

| Concept | Meaning |
|---|---|
| Database | A collection of related tables |
| Table | Data arranged in rows and columns |
| Row | One record or observation |
| Column | One field, such as `area` or `taxi_count` |
| Value | One cell in a row/column |
| Primary key | A column, or set of columns, that identifies a row |
| Foreign key | A column that refers to a key in another table |

For example, `dim_area.area_id` can identify one area. A taxi fact row can store that `area_id` to point back to the area table.

## 2. ETL, ELT, and Auditability

There are two common patterns.

| Pattern | Meaning | Tradeoff |
|---|---|---|
| ETL | Extract, Transform, Load | Clean data before loading into the database |
| ELT | Extract, Load, Transform | Load data first, then transform with SQL inside the database |

In Step 02, we did a lot of transformation before loading. That is close to ETL.

In this notebook, we also create staging tables. A staging table keeps the loaded CSV data before we build the final model tables.

Why keep staging tables?

- We can audit where data came from.
- We can rerun a transformation if our logic changes.
- We avoid losing useful source columns too early.
- We can compare staged data and final tables.

## 3. Environment

DuckDB is a database that runs inside Python. No separate database server is needed.

In [4]:
from pathlib import Path

import duckdb
import pandas as pd


def find_day1_base_dir():
    current = Path.cwd().resolve()

    if (current / "day_1.pptx").exists():
        return current

    if (current / "day_1" / "day_1.pptx").exists():
        return current / "day_1"

    for parent in current.parents:
        if parent.name == "day_1" or (parent / "day_1.pptx").exists():
            return parent

    return current


BASE_DIR = find_day1_base_dir()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
SHARED_DATA_DIR = BASE_DIR / "shared_data"
SHARED_DATA_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = SHARED_DATA_DIR / "day_1_weather_taxi_data.duckdb"

weather_csv = PROCESSED_DIR / "clean_weather_30min_by_area.csv"
taxi_csv = PROCESSED_DIR / "clean_taxi_30min_by_area.csv"
rainfall_csv = PROCESSED_DIR / "clean_rainfall_30min_by_area.csv"

print("Database:", DB_PATH)
print("Weather CSV:", weather_csv)
print("Taxi CSV:", taxi_csv)
print("Rainfall CSV:", rainfall_csv)

Database: /Users/dlai/dev_repo/AI-projects/SNAIC_AI_Program_Labs/Week 2/Day 1/shared_data/day_1_weather_taxi_data.duckdb
Weather CSV: /Users/dlai/dev_repo/AI-projects/SNAIC_AI_Program_Labs/Week 2/Day 1/data/processed/clean_weather_30min_by_area.csv
Taxi CSV: /Users/dlai/dev_repo/AI-projects/SNAIC_AI_Program_Labs/Week 2/Day 1/data/processed/clean_taxi_30min_by_area.csv
Rainfall CSV: /Users/dlai/dev_repo/AI-projects/SNAIC_AI_Program_Labs/Week 2/Day 1/data/processed/clean_rainfall_30min_by_area.csv


## 4. Preview the Clean CSV Files

Before loading into a database, always check the columns and a few rows.

In [5]:
weather_df = pd.read_csv(weather_csv)
taxi_df = pd.read_csv(taxi_csv)
rainfall_df = pd.read_csv(rainfall_csv)

print("weather", weather_df.shape)
print("taxi", taxi_df.shape)
print("rainfall", rainfall_df.shape)

weather_df.head(3)

weather (47, 6)
taxi (52, 3)
rainfall (39, 3)


,timestamp_30min,record_timestamp,forecast_start_timestamp,forecast_end_timestamp,area,weather_forecast
0,2026-06-22 11:00:00,2026-06-22 11:00:00,2026-06-22 11:00:00,2026-06-22 13:00:00,Ang Mo Kio,Showers
1,2026-06-22 11:00:00,2026-06-22 11:00:00,2026-06-22 11:00:00,2026-06-22 13:00:00,Bedok,Thundery Showers
2,2026-06-22 11:00:00,2026-06-22 11:00:00,2026-06-22 11:00:00,2026-06-22 13:00:00,Bishan,Showers


In [6]:
taxi_df.head(3)

,timestamp_30min,area,taxi_count
0,2026-06-22 11:00:00,Ang Mo Kio,139
1,2026-06-22 11:00:00,Bedok,122
2,2026-06-22 11:00:00,Bishan,48


In [7]:
rainfall_df.head(3)

,timestamp_30min,area,rainfall_mm
0,2026-06-22 11:00:00,Ang Mo Kio,0.0
1,2026-06-22 11:00:00,Bedok,1.2
2,2026-06-22 11:00:00,Bishan,0.2


## 5. Connect to DuckDB

This creates a local database file. If it already exists, DuckDB opens it.

In [8]:
con = duckdb.connect(DB_PATH)
con.execute("SELECT 1 AS connected").df()

,connected
0,1


## 6. Mini Practice: Create Tables by Hand

Before loading a real CSV file, start with the normal database pattern:

1. Create a table.
2. Define the columns and data types.
3. Insert rows.
4. Query the table.

This small example uses two tables:

- `mini_area`: one row per area
- `mini_weather`: one weather record per timestamp and area

`mini_area.area_id` is a primary key. It identifies one area.

`mini_weather.area_id` is a foreign key. It points back to `mini_area.area_id`.

In [9]:
con.execute("DROP TABLE IF EXISTS mini_weather")
con.execute("DROP TABLE IF EXISTS mini_area")

con.execute("""
CREATE TABLE mini_area (
    area_id INTEGER PRIMARY KEY,
    area VARCHAR NOT NULL
)
""")

con.execute("""
CREATE TABLE mini_weather (
    timestamp_30min TIMESTAMP,
    area_id INTEGER,
    weather_forecast VARCHAR,
    PRIMARY KEY (timestamp_30min, area_id),
    FOREIGN KEY (area_id) REFERENCES mini_area(area_id)
)
""")

con.execute("DESCRIBE mini_weather").df()

,column_name,column_type,null,key,default,extra
0,timestamp_30min,TIMESTAMP,NO,PRI,None,None
1,area_id,INTEGER,NO,PRI,None,None
2,weather_forecast,VARCHAR,YES,None,None,None


## 7. Mini Practice: Insert Rows and Query

For a tiny example, we can insert rows manually with `INSERT INTO ... VALUES ...`.

In real pipelines, we usually load many rows from CSV, Parquet, a DataFrame, or another database table.

In [12]:
con.execute("""
DELETE FROM mini_weather
""")
con.execute("""
DELETE FROM mini_area
""")

con.execute("""
INSERT INTO mini_area VALUES
    (1, 'Ang Mo Kio'),
    (2, 'Bedok'),
    (3, 'Bishan')
""")

con.execute("""
INSERT INTO mini_weather VALUES
    ('2026-06-07 22:00:00', 1, 'Partly Cloudy (Night)'),
    ('2026-06-07 22:00:00', 2, 'Partly Cloudy (Night)'),
    ('2026-06-07 22:00:00', 3, 'Partly Cloudy (Night)')
""")

con.execute("SELECT * FROM mini_area ORDER BY area_id").df()

,area_id,area
0,1,Ang Mo Kio
1,2,Bedok
2,3,Bishan


The `mini_weather` table stores `area_id`, not the area name.

To show the area name, join `mini_weather` to `mini_area`.

In [13]:
con.execute("""
SELECT
    w.timestamp_30min,
    a.area,
    w.weather_forecast
FROM mini_weather AS w
JOIN mini_area AS a
  ON w.area_id = a.area_id
ORDER BY a.area
""").df()

,timestamp_30min,area,weather_forecast
0,2026-06-07 22:00:00,Ang Mo Kio,Partly Cloudy (Night)
1,2026-06-07 22:00:00,Bedok,Partly Cloudy (Night)
2,2026-06-07 22:00:00,Bishan,Partly Cloudy (Night)


## 8. Advanced Shortcut: Load CSV Files into Staging Tables

Now that we have seen normal `CREATE TABLE` and `INSERT`, we can use a DuckDB shortcut.

`CREATE TABLE ... AS SELECT ... FROM read_csv_auto(...)` creates a table from a CSV-reading query.

This is convenient, but it is easier to understand after seeing the manual version first.


A staging table is close to the loaded source file.

We keep these tables for auditability. If a final query looks wrong, we can trace back to the staged data.

In [14]:
con.execute("DROP TABLE IF EXISTS staging_weather")
con.execute("DROP TABLE IF EXISTS staging_taxi")
con.execute("DROP TABLE IF EXISTS staging_rainfall")

con.execute("""
CREATE TABLE staging_weather AS
SELECT
    CAST(timestamp_30min AS TIMESTAMP) AS timestamp_30min,
    CAST(record_timestamp AS TIMESTAMP) AS record_timestamp,
    CAST(forecast_start_timestamp AS TIMESTAMP) AS forecast_start_timestamp,
    CAST(forecast_end_timestamp AS TIMESTAMP) AS forecast_end_timestamp,
    area,
    weather_forecast
FROM read_csv_auto(?)
""", [str(weather_csv)])

con.execute("""
CREATE TABLE staging_taxi AS
SELECT
    CAST(timestamp_30min AS TIMESTAMP) AS timestamp_30min,
    area,
    CAST(taxi_count AS INTEGER) AS taxi_count
FROM read_csv_auto(?)
""", [str(taxi_csv)])

con.execute("""
CREATE TABLE staging_rainfall AS
SELECT
    CAST(timestamp_30min AS TIMESTAMP) AS timestamp_30min,
    area,
    CAST(rainfall_mm AS DOUBLE) AS rainfall_mm
FROM read_csv_auto(?)
""", [str(rainfall_csv)])

con.execute("""
CREATE OR REPLACE TABLE import_log AS
SELECT 'weather' AS source_name, ? AS source_file, COUNT(*) AS row_count, now() AS loaded_at FROM staging_weather
UNION ALL
SELECT 'taxi', ?, COUNT(*), now() FROM staging_taxi
UNION ALL
SELECT 'rainfall', ?, COUNT(*), now() FROM staging_rainfall
""", [weather_csv.name, taxi_csv.name, rainfall_csv.name])

con.execute("SELECT * FROM import_log ORDER BY source_name").df()

,source_name,source_file,row_count,loaded_at
0,rainfall,clean_rainfall_30min_by_area.csv,39,2026-06-22 13:42:28.037704+08:00
1,taxi,clean_taxi_30min_by_area.csv,52,2026-06-22 13:42:28.037704+08:00
2,weather,clean_weather_30min_by_area.csv,47,2026-06-22 13:42:28.037704+08:00


## 9. Inspect Tables and Schemas

A schema tells us the table structure: column names and data types.

In [15]:
con.execute("SHOW TABLES").df()

,name
0,import_log
1,mini_area
2,mini_weather
3,staging_rainfall
4,staging_taxi
5,staging_weather


In [16]:
con.execute("DESCRIBE staging_weather").df()

,column_name,column_type,null,key,default,extra
0,timestamp_30min,TIMESTAMP,YES,None,None,None
1,record_timestamp,TIMESTAMP,YES,None,None,None
2,forecast_start_timestamp,TIMESTAMP,YES,None,None,None
3,forecast_end_timestamp,TIMESTAMP,YES,None,None,None
4,area,VARCHAR,YES,None,None,None
5,weather_forecast,VARCHAR,YES,None,None,None


## 10. Create Dimension and Fact Tables

A common database design is to separate dimensions and facts.

| Table type | Meaning | Example |
|---|---|---|
| Dimension | Descriptive lookup table | `dim_area` |
| Fact | Measurements or events | taxi count, rainfall mm, weather forecast |

Here, `dim_area` gives each area an `area_id`. The fact tables store `area_id` as a foreign-key-like reference.

In [ ]:
con.execute("DROP TABLE IF EXISTS fact_weather")
con.execute("DROP TABLE IF EXISTS fact_taxi")
con.execute("DROP TABLE IF EXISTS fact_rainfall")
con.execute("DROP TABLE IF EXISTS dim_area")

con.execute("""
CREATE TABLE dim_area AS
SELECT
    row_number() OVER (ORDER BY area) AS area_id,
    area
FROM (
    SELECT DISTINCT area FROM staging_weather
    UNION
    SELECT DISTINCT area FROM staging_taxi
    UNION
    SELECT DISTINCT area FROM staging_rainfall
)
""")

con.execute("""
CREATE TABLE fact_weather AS
SELECT
    w.timestamp_30min,
    w.record_timestamp,
    w.forecast_start_timestamp,
    w.forecast_end_timestamp,
    a.area_id,
    w.weather_forecast
FROM staging_weather AS w
JOIN dim_area AS a
  ON w.area = a.area
""")

con.execute("""
CREATE TABLE fact_taxi AS
SELECT
    t.timestamp_30min,
    a.area_id,
    t.taxi_count
FROM staging_taxi AS t
JOIN dim_area AS a
  ON t.area = a.area
""")

con.execute("""
CREATE TABLE fact_rainfall AS
SELECT
    r.timestamp_30min,
    a.area_id,
    r.rainfall_mm
FROM staging_rainfall AS r
JOIN dim_area AS a
  ON r.area = a.area
""")

con.execute("SELECT * FROM dim_area ORDER BY area_id LIMIT 10").df()

,area_id,area
0,1,Ang Mo Kio
1,2,Bedok
2,3,Bishan
3,4,Boon Lay
4,5,Bukit Batok
5,6,Bukit Merah
6,7,Bukit Panjang
7,8,Bukit Timah
8,9,Central Water Catchment
9,10,Changi


## 11. Basic SQL: SELECT, WHERE, ORDER BY, LIMIT

`SELECT` chooses columns. `WHERE` filters rows. `ORDER BY` sorts. `LIMIT` keeps a small number of rows.

In [18]:
con.execute("""
SELECT
    t.timestamp_30min,
    a.area,
    t.taxi_count
FROM fact_taxi AS t
JOIN dim_area AS a
  ON t.area_id = a.area_id
WHERE t.taxi_count >= 100
ORDER BY t.taxi_count DESC, a.area
LIMIT 10
""").df()

,timestamp_30min,area,taxi_count
0,2026-06-22 11:00:00,Changi,337
1,2026-06-22 11:00:00,Bukit Merah,291
2,2026-06-22 11:00:00,Tampines,171
3,2026-06-22 11:00:00,Queenstown,162
4,2026-06-22 11:00:00,Downtown Core,158
5,2026-06-22 11:00:00,Kallang,143
6,2026-06-22 11:00:00,Ang Mo Kio,139
7,2026-06-22 11:00:00,Geylang,137
8,2026-06-22 11:00:00,Bedok,122
9,2026-06-22 11:00:00,Jurong West,116


## 12. Aggregation: GROUP BY

Aggregation summarizes many rows into fewer rows.

Common aggregate functions:

| Function | Meaning |
|---|---|
| `COUNT(*)` | Count rows |
| `SUM()` | Add values |
| `AVG()` | Average values |
| `MIN()` | Smallest value |
| `MAX()` | Largest value |

Example question: which areas have the most taxis?

In [ ]:
con.execute("""
SELECT
    a.area,
    SUM(t.taxi_count) AS total_taxis,
    AVG(t.taxi_count) AS average_taxis
FROM fact_taxi AS t
JOIN dim_area AS a
  ON t.area_id = a.area_id
GROUP BY a.area
ORDER BY total_taxis DESC
LIMIT 10
""").df()

## 13. WHERE vs HAVING

`WHERE` filters rows before aggregation.

`HAVING` filters groups after aggregation.

Example question: after grouping by area, keep only areas with at least 50 taxis.

In [ ]:
con.execute("""
SELECT
    a.area,
    SUM(t.taxi_count) AS total_taxis
FROM fact_taxi AS t
JOIN dim_area AS a
  ON t.area_id = a.area_id
GROUP BY a.area
HAVING SUM(t.taxi_count) >= 50
ORDER BY total_taxis DESC
""").df()

## 14. Joins

A join combines rows from two tables.

| Join type | Meaning |
|---|---|
| `INNER JOIN` | Keep matched rows only |
| `LEFT JOIN` | Keep all rows from the left table, plus matched right rows |
| `RIGHT JOIN` | Keep all rows from the right table, plus matched left rows |
| `FULL JOIN` | Keep all rows from both tables, matched where possible |

A join can happen during transformation before loading into a database. But if we join too early and save only the merged output, we may lose flexibility.

In the database, we can keep weather, taxi, and rainfall as separate tables, then join them differently depending on the question.

In [ ]:
con.execute("""
SELECT
    w.timestamp_30min,
    a.area,
    w.weather_forecast,
    t.taxi_count,
    r.rainfall_mm
FROM fact_weather AS w
JOIN dim_area AS a
  ON w.area_id = a.area_id
LEFT JOIN fact_taxi AS t
  ON w.area_id = t.area_id
 AND w.timestamp_30min = t.timestamp_30min
LEFT JOIN fact_rainfall AS r
  ON w.area_id = r.area_id
 AND w.timestamp_30min = r.timestamp_30min
ORDER BY t.taxi_count DESC NULLS LAST, a.area
LIMIT 15
""").df()

## 15. Join Audit: What Did Not Match?

A `LEFT JOIN` can reveal missing matches.

Example question: which weather areas do not have taxi data in this 30-minute snapshot?

In [ ]:
con.execute("""
SELECT
    a.area,
    w.weather_forecast,
    t.taxi_count
FROM fact_weather AS w
JOIN dim_area AS a
  ON w.area_id = a.area_id
LEFT JOIN fact_taxi AS t
  ON w.area_id = t.area_id
 AND w.timestamp_30min = t.timestamp_30min
WHERE t.area_id IS NULL
ORDER BY a.area
""").df()

## 16. NULL and COUNT

`NULL` means unknown or missing.

`COUNT(*)` counts all rows.

`COUNT(column_name)` counts only non-NULL values in that column.

In [ ]:
con.execute("""
WITH joined AS (
    SELECT
        a.area,
        w.weather_forecast,
        t.taxi_count,
        r.rainfall_mm
    FROM fact_weather AS w
    JOIN dim_area AS a
      ON w.area_id = a.area_id
    LEFT JOIN fact_taxi AS t
      ON w.area_id = t.area_id
     AND w.timestamp_30min = t.timestamp_30min
    LEFT JOIN fact_rainfall AS r
      ON w.area_id = r.area_id
     AND w.timestamp_30min = r.timestamp_30min
)
SELECT
    COUNT(*) AS all_weather_area_rows,
    COUNT(taxi_count) AS rows_with_taxi_count,
    COUNT(rainfall_mm) AS rows_with_rainfall_mm
FROM joined
""").df()

## 17. Auditability Queries

Auditability means we can answer questions such as:

- Which files were loaded?
- How many rows did each file produce?
- Do staging row counts match final table row counts?
- Are there duplicate keys?
- Are there missing joined values?

In [ ]:
con.execute("SELECT * FROM import_log ORDER BY source_name").df()

In [ ]:
con.execute("""
SELECT 'staging_weather' AS table_name, COUNT(*) AS row_count FROM staging_weather
UNION ALL
SELECT 'fact_weather', COUNT(*) FROM fact_weather
UNION ALL
SELECT 'staging_taxi', COUNT(*) FROM staging_taxi
UNION ALL
SELECT 'fact_taxi', COUNT(*) FROM fact_taxi
UNION ALL
SELECT 'staging_rainfall', COUNT(*) FROM staging_rainfall
UNION ALL
SELECT 'fact_rainfall', COUNT(*) FROM fact_rainfall
ORDER BY table_name
""").df()

## 18. Save an Analytical View

Instead of permanently merging everything into one CSV, we can create a database view.

A view stores the query logic. The source tables remain separate.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW dashboard_area_snapshot AS
SELECT
    w.timestamp_30min,
    a.area,
    w.weather_forecast,
    t.taxi_count,
    r.rainfall_mm
FROM fact_weather AS w
JOIN dim_area AS a
  ON w.area_id = a.area_id
LEFT JOIN fact_taxi AS t
  ON w.area_id = t.area_id
 AND w.timestamp_30min = t.timestamp_30min
LEFT JOIN fact_rainfall AS r
  ON w.area_id = r.area_id
 AND w.timestamp_30min = r.timestamp_30min
""")

con.execute("SELECT * FROM dashboard_area_snapshot ORDER BY taxi_count DESC NULLS LAST LIMIT 10").df()

## 19. Close the Database Connection

When a notebook keeps a DuckDB connection open, the database file may stay locked by the Jupyter kernel.

Before moving to the serving step, checkpoint the database and close the connection.

In [ ]:
con.execute("CHECKPOINT")
con.close()

print("DuckDB connection closed.")

## 20. Review Questions

**Q1. What is a relational database?**

A relational database stores data in related tables made of rows and columns.

**Q2. What is the difference between a row, column, and value?**

A row is one record. A column is one field. A value is one cell in a row and column.

**Q3. What is a primary key?**

A primary key identifies one row in a table.

**Q4. What is a foreign key?**

A foreign key points to a key in another table.

**Q5. What is aggregation?**

Aggregation summarizes multiple rows, using functions such as `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX`.

**Q6. What is a join?**

A join combines rows from two tables using a matching condition.

**Q7. Why keep staging tables?**

Staging tables help audit the data source, rerun transformations, and avoid losing information too early.

**Q8. Explain the differences among INNER JOIN, LEFT JOIN, RIGHT JOIN, and FULL JOIN.**

INNER JOIN returns matched rows only. LEFT JOIN returns all rows from the left table plus matched right rows. RIGHT JOIN is the opposite. FULL JOIN returns all rows from both tables, matched where possible.

**Q9. What is the difference between WHERE and HAVING?**

WHERE filters rows before aggregation. HAVING filters groups after aggregation. HAVING is used with aggregate conditions such as `COUNT(*) > 1`.

**Q10. How does SQL handle NULL values? What is the difference between COUNT(*) and COUNT(column_name)?**

NULL means unknown or missing. COUNT(*) counts all rows, while COUNT(column_name) counts only non-NULL values in that column.

## 21. Stop Here

The storage flow is:

```text
clean CSV -> staging tables -> dimension/fact tables -> SQL aggregation and joins -> analytical view
```

This keeps the project flexible. We can join tables for one question, aggregate for another question, and still trace the result back to the loaded files.